In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [4]:
df_new = read_csv("../data/raw/stop_data_2024_current.csv")
# df_new

New names:
• `` -> `...58`
Warning message:
"One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)"
Rows: 10931 Columns: 58
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (40): LEA_Record_ID, StopDate, CallForService, WelfareWellnessCheck, In...
dbl   (9): PersonNum, StopDuration, Latitude, Longitude, BlockNumber, Highwa...
lgl   (8): OtherLocationDesc, IsK12School, SchoolName, StopForAStudent, Traf...
time  (1): StopTime

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
df_new |>
  mutate(StopDate = mdy(StopDate)) |>
  # summarise(min_date = min(StopDate), max_date = max(StopDate))
  # distinct(City)
  filter(is.na(Longitude)) # because these are so few we could just exclude from the analysis for ease of analysis

LEA_Record_ID,PersonNum,StopDate,StopTime,StopDuration,CallForService,WelfareWellnessCheck,InfoBasedStop,TypeOfStopDesc,LocationType,⋯,WeaponsFound,ForceActionsTaken,ContrabandOrEvidence,ResultOfStop,VerbalWarningOffense,WrittenWarningOffense,CitationForInfractionOffense,InfieldCiteRelOffense,CustodialArrestNoWarrantOffense,...58
<chr>,<dbl>,<date>,<time>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<lgl>
B00000000037,1,2024-01-01,19:33:00,40,N,N,Y,Pedestrian Stop,BlkNumber,⋯,No,Handcuffed or flex cuffed,None,Custodial arrest pursuant to outstanding warrant,NA,NA,NA,NA,NA,NA
B00000000054,1,2024-01-02,02:16:00,20,Y,N,N,Pedestrian Stop,BlkNumber,⋯,No,Handcuffed or flex cuffed,None,No action,NA,NA,NA,NA,NA,NA
B00000000763,1,2024-03-05,11:19:00,30,Y,N,Y,Pedestrian Stop,BlkNumber,⋯,No,Handcuffed or flex cuffed,None,Custodial arrest pursuant to outstanding warrant,NA,NA,NA,NA,NA,NA
B00000000774,1,2024-03-06,09:40:00,40,Y,Y,Y,Pedestrian Stop,BlkNumber,⋯,No,Handcuffed or flex cuffed,None,Psychiatric hold (W&I Code 5150 or 5585.20),NA,NA,NA,NA,NA,NA
B00000000951,1,2024-03-19,23:12:00,10,N,N,N,Vehicular Stop,Intersection,⋯,No,None,None,Verbal warning,VC 4000(A)(1) NO REG:VEH/TRAILER/ETC (I),NA,NA,NA,NA,NA
B00000001057,1,2024-03-25,18:40:00,30,Y,N,Y,Pedestrian Stop,BlkNumber,⋯,No,Handcuffed or flex cuffed,None,Custodial arrest without warrant,NA,NA,NA,NA,"PC 484(A) PETTY THEFT (M),PC 484(A) PETTY THEFT (M)",NA
B00000001261,1,2024-04-08,22:38:00,30,Y,N,Y,Vehicular Stop,BlkNumber,⋯,No,Handcuffed or flex cuffed,None,No action,NA,NA,NA,NA,NA,NA
B00000001282,1,2024-04-10,08:00:00,8,N,N,N,Vehicular Stop,Intersection,⋯,No,None,None,Citation for infraction,NA,NA,VC 21655.5(B) FAIL OBEY TRAF LANE SIGN (I),NA,NA,NA
B00000001283,1,2024-04-10,08:08:00,8,N,N,N,Vehicular Stop,HighwayAndExit,⋯,No,None,None,Citation for infraction,NA,NA,VC 21655.5(B) FAIL OBEY TRAF LANE SIGN (I)|VC 27315(D)(1) SEATBELT VIOLATION (I),NA,NA,NA


In [7]:
df_old = read_csv("../data/raw/stop_data_2020_2023.csv")
# df_old

Warning message:
"One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)"
Rows: 18090 Columns: 44
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (37): LEA_RecordID, DateTimeOfStop, RespToServiceCall, InfoBasedStop, Ty...
dbl  (6): PersonNum, DateTimeSortable, DurationOfStop, Latitude, Longitude, Age
lgl  (1): GenderNonconforming

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [8]:
df_old |>
  mutate(DateTimeOfStop = mdy_hm(DateTimeOfStop)) |>
  # summarise(min_date = min(DateTimeOfStop), max_date = max(DateTimeOfStop))
  # filter(is.na(Longitude)) # none
  distinct(TrafficViolationType)

TrafficViolationType
<chr>
NA
Equipment violation
Moving violation
Non-moving violation including registration


In [9]:
df_old_cleaned = df_old |>
  mutate(
    DateTimeOfStop = mdy_hm(DateTimeOfStop),
    RespToServiceCall = ifelse(RespToServiceCall == "Yes", "Y", "N"),
    TypeOfStop = ifelse(TypeOfStop == "Vehicle", "Vehicular Stop", str_c(TypeOfStop, " Stop")),
    LocationType = "LatLonPt",
  ) |>
  filter(
    year(DateTimeOfStop) >= 2021, year(DateTimeOfStop) <= 2023,
    City == "Berkeley",
  ) |>
  select(LEA_Record_ID = LEA_RecordID, PersonNum, DateTimeOfStop, 
    CallForService = RespToServiceCall, TypeOfStopDesc = TypeOfStop, StopDuration = DurationOfStop,
    City, LocationType, Latitude, Longitude, 
    LocationDescription
  )

df_old_cleaned |>
  summarise(min_date = min(DateTimeOfStop), max_date = max(DateTimeOfStop))

min_date,max_date
<dttm>,<dttm>
2021-01-01 01:18:00,2023-12-31 20:06:00


In [10]:
df_new_cleaned = df_new |>
  mutate(
    DateTimeOfStop = mdy_hms(str_c(StopDate, " ", StopTime)),
    City = str_to_title(City)
  ) |>
  filter(
    City == "Berkeley"
  ) |> 
  select(LEA_Record_ID, PersonNum, DateTimeOfStop, 
    CallForService, TypeOfStopDesc, StopDuration,
    City, LocationType, Latitude, Longitude,
    BlockNumber, StreetName, CrossStreet1, CrossStreet2, Highway, ClosestExit, OtherLocationDesc
  )

df_new_cleaned |>
  summarise(min_date = min(DateTimeOfStop), max_date = max(DateTimeOfStop))
# do we want to filter out half-2026 data?

min_date,max_date
<dttm>,<dttm>
2024-01-01 01:20:00,2026-05-26 22:28:00


In [12]:
df = full_join(df_old_cleaned, df_new_cleaned)

write_csv(df, "../data/clean/stop_data.csv")

Joining with `by = join_by(LEA_Record_ID, PersonNum, DateTimeOfStop,
CallForService, TypeOfStopDesc, StopDuration, City, LocationType, Latitude,
Longitude)`
